# Handling Class Imbalance with Weighted Loss
In medical datasets, healthy patients usually heavily outnumber sick patients. This notebook demonstrates how to calculate label frequencies and apply a "Weighted Loss Function" to prevent an AI from lazily guessing "healthy" on every scan.

Step 1: Counting the Diseases

What it does: We load the spreadsheet and just count the number of sick people for each disease.

In [2]:
import pandas as pd

# Load the hospital data
hospital_data = pd.read_csv("/content/train-small.csv")

# Remove the ID columns so we only look at diseases
disease_data = hospital_data.drop(columns=['Image', 'PatientId'])

# Add up the 1s to see how many sick people we have per disease
sick_counts = disease_data.sum()

print(sick_counts)

Atelectasis           106
Cardiomegaly           20
Consolidation          33
Edema                  16
Effusion              128
Emphysema              13
Fibrosis               14
Hernia                  2
Infiltration          175
Mass                   45
Nodule                 54
Pleural_Thickening     21
Pneumonia              10
Pneumothorax           38
dtype: int64


Code Pattern: Real Data Ingestion & Counting.

Step 2: The Unfair Grader (Regular Loss on Real Data)

What it does: We take the entire grid of real patient data. We simulate a lazy AI that guesses every single patient is 90% sick. We use the standard log formula to calculate the error. Because your real data is heavily imbalanced (mostly healthy people), the AI will get a massive penalty for the healthy patients and a tiny penalty for the rare sick patients.

In [4]:
import numpy as np

# Convert the pandas table into a numpy math grid
actual_patients = disease_data.values

# Our lazy AI guesses 0.9 (90% sick) for every patient and every disease
ai_guesses = 0.9 * np.ones(actual_patients.shape)

# The Grader calculates the raw error for the sick patients
error_for_sick = -np.sum(actual_patients * np.log(ai_guesses))

# The Grader calculates the raw error for the healthy patients
error_for_healthy = -np.sum((1 - actual_patients) * np.log(1 - ai_guesses))

total_error = error_for_sick + error_for_healthy
print(f"Total Error with Unfair Grader: {total_error:.4f}")

Total Error with Unfair Grader: 30753.0647


Step 3: The Fair Grader (Weighted Loss for a Single Disease)

What it does: Let's focus on just one real disease from your file, like the "Mass" column. We calculate the weights dynamically based on the exact number of real patients in that specific column. Then we apply those weights to fix the grader for that one disease.

In [5]:
# Isolate just ONE disease column to see the fair grader in action
actual_mass_patients = disease_data['Mass'].values
ai_mass_guesses = 0.9 * np.ones(actual_mass_patients.shape)

# Calculate weights based on the REAL data size for this column
total_patients = len(actual_mass_patients)
weight_for_sick = np.sum(actual_mass_patients == 0) / total_patients
weight_for_healthy = np.sum(actual_mass_patients == 1) / total_patients

# Multiply the old errors by the new weights
fair_error_sick = -np.sum(weight_for_sick * actual_mass_patients * np.log(ai_mass_guesses))
fair_error_healthy = -np.sum(weight_for_healthy * (1 - actual_mass_patients) * np.log(1 - ai_mass_guesses))

total_fair_error = fair_error_sick + fair_error_healthy
print(f"Fair Error for 'Mass' column: {total_fair_error:.4f}")

Fair Error for 'Mass' column: 103.4815


Code Pattern: Dynamic Single-Column Weighting.

Step 4: Multi-Tasking (Checking ALL Real Diseases at Once)

What it does: In a real hospital setting, we do not want to write Step 3 fourteen times for fourteen different diseases! We use the entire 2D grid of real data and use axis=0 to tell Python to calculate the weights for every single disease column simultaneously.

In [6]:
# Using the full grid from Step 2
actual_patients = disease_data.values
ai_guesses = 0.9 * np.ones(actual_patients.shape)

# axis=0 tells the computer: "Look DOWN the columns" to calculate weights for every disease at once
weights_for_sick_multitask = np.sum(actual_patients == 0, axis=0) / actual_patients.shape[0]
weights_for_healthy_multitask = np.sum(actual_patients == 1, axis=0) / actual_patients.shape[0]

# Calculate the final multi-task loss using the matrix of weights
loss_pos_multi = -np.sum(weights_for_sick_multitask * actual_patients * np.log(ai_guesses))
loss_neg_multi = -np.sum(weights_for_healthy_multitask * (1 - actual_patients) * np.log(1 - ai_guesses))

total_multi_loss = loss_pos_multi + loss_neg_multi
print(f"Total Multi-Task Weighted Loss: {total_multi_loss:.4f}")

Total Multi-Task Weighted Loss: 1463.3447


## Conclusion: The Power of Weighted Loss

This notebook illustrated the critical problem of **class imbalance** in medical datasets, where a naive AI trained with a standard loss function (the 'Unfair Grader') would be heavily penalized by the abundant healthy cases and overlook the rarer, but clinically important, sick cases.

By dynamically calculating **weights** based on disease frequencies and applying a **weighted loss function** (the 'Fair Grader'), we demonstrated how to rebalance the impact of healthy and sick patients on the model's error. This ensures that the AI is adequately penalized for misclassifying rare diseases, leading to more robust and clinically relevant models. Finally, the notebook showed how to extend this concept to a **multi-task weighted loss**, efficiently handling all diseases simultaneously without repetitive code, which is essential for real-world applications.